# Analysis for the counter-zero tests

### This is to deal with MWorks skipping counter zero after code changes in ngfriedman/2310/imaginggate-laseroff 

- files in data/241111-countertest-issue[123].mwk2
- testing mwkfiles.py reader code

### references
- see MH LabArchives notes

# Logic of changes

### Previously, in mwkfiles.py

- Find the last zero tick
- Find the first 1 tick
- Check (line 372, 1e076ce) that the stim level is set between the 0 and 1 tick. (This no longer happens)
- Check that no counter ticks are skipped or duplicated
- Some rough checks that we have the right number of ticks for the number of stimuli

### What changed in hardware

- In some cases on rig C2, when scanimage is started, resetting the MWorks counter variable sets the counter to 1 —-- the 1 tick happens in under 1ms after the 0 tick

### Now, plan for what to do:
- we detect this situation and subtract 1 from all counter values.
- key check: if first 1 (in 4th or 5th index spot) follows previous code by < 1ms, it means the zero code was missed, the first 1 is really zero, and the first 2 is really a 1. Fix by subtracting.


## TODO 250119:
- looks like the problem is in finding the start of the trial - with counter tick = 1 or something else. Let's figure out where the counter 1 happens and the first tick time and see if that needs correcting or can be used as the start of the trial, or if I have to find the start of the trial a different way
- Smaller h5 files for push

In [1]:
%autoreload 2

from pathlib import Path
import sys, os
from types import SimpleNamespace
import pandas as pd 

import mworksbehavior as mwb
from mworksbehavior import mwkfiles

## check to make sure the mwkfiles code is coming from the right place
# there may be better ways to do this, by forcing mwb directory to
# start of sys.path, for ex
packdir = Path('~/Repositories/mworksbehavior/mworksbehavior').expanduser()
# if this assertion fails you may need cd ~/Repositories/mworksbehavior; pip -e .
assert packdir.resolve() == Path(mwb.__path__[0]).resolve(), \
  'does not look like mworksbehavior is installed in editable form at ~/Repositories/mworksbehavior'

In [2]:
datdir = Path('~/Repositories/mworksbehavior/mworksbehavior/tests/data').expanduser()

#mwkname = f'240918-countertest-issue1.mwk2'
mwkname = f'240918-countertest-issue2.mwk2'
#mwkname = f'241111-countertest-issue3.mwk2'


### Read the file using the plain MWF code and display the codestream for debugging

In [3]:
# Read the file using the plain MWF code
fulln = datdir / mwkname

mwf = mwkfiles.MWKFile(fulln);


In [4]:
print(mwf.firstTrStartTimeUs)
print(mwf.lastTrEndTimeUs)

262011259
330127328


In [5]:
counterIx = mwf.df.tagname.isin(['stimPySelLevel'])
mwf.df.loc[counterIx,:]

,tagname,timeUs,value,timeFmStUs
159,stimPySelLevel,260282108,2,129
363,stimPySelLevel,260284031,2,2052
574,stimPySelLevel,262000699,2,1718720
693,stimPySelLevel,262016905,2,1734926
1922,stimPySelLevel,274133434,5,13851455
2905,stimPySelLevel,282133522,6,21851543
3888,stimPySelLevel,290134206,7,29852227
4871,stimPySelLevel,298133377,0,37851398
5866,stimPySelLevel,306133609,4,45851630
6849,stimPySelLevel,314133860,3,53851881


In [6]:
## quick counterFIO analysis from raw stream
counterIx = mwf.df.tagname.isin(['counterFIO'])
cdf = mwf.df.loc[counterIx,:]
np.unique(np.diff(cdf.value))

array([-2244, 0, 1], dtype=object)

In [7]:
counterIx = mwf.df.tagname.isin(['counterFIO','strobedDigitalWord', 'stimPySelLevel'])
pd.set_option('display.max_rows', 60)
pd.set_option('display.min_rows', 50)


mwf.df.loc[counterIx,:]

,tagname,timeUs,value,timeFmStUs
159,stimPySelLevel,260282108,2,129
197,strobedDigitalWord,260282124,4,145
199,counterFIO,260282124,2245,145
363,stimPySelLevel,260284031,2,2052
401,counterFIO,260284135,2245,2156
402,strobedDigitalWord,260284136,4,2157
574,stimPySelLevel,262000699,2,1718720
612,counterFIO,262000750,2245,1718771
613,strobedDigitalWord,262000750,4,1718771
637,strobedDigitalWord,262009570,0,1727591


### Test with the full mwkfiles.RetinotopyMap2Stim / CounterStimMixin parsing code

In [8]:
mwf = mwkfiles.RetinotopyMap2StimMWKFile(fulln,doTryFixCorrupt=True)
mwf.compute_imaging_constants()
print(mwf.nstim, mwf.nframes_stim)
assert mwf.nstim * mwf.nframes_stim == 1920

262016795
1919 12 159.91666666666666
8 240


/Users/histed/Repositories/mworksbehavior/mworksbehavior/mwkfiles.py:374: UserWarning: Found first counter tick to be 1, expected zero; subtracting 1 from all counter vals, altering df values
  warnings.warn(f"Found first counter tick to be 1, expected zero; subtracting 1 from all counter vals, altering df values")
/Users/histed/Repositories/mworksbehavior/mworksbehavior/mwkfiles.py:474: UserWarning: Trying to fix file: Number of frames is one less than expected, probably first trial is short, be careful w/ analysis
  warnings.warn('Trying to fix file: Number of frames is one less than expected, probably first trial is short, be careful w/ analysis')


In [14]:
display(mwf.counterStats)

namespace(nCounterTicks=1919, nTotalStims=12)

In [9]:
display(mwf.firstCounterUs)
display(mwf.firstTrStartTimeUs)
display(mwf.firstTrEndTimeUs)
display(mwf.constS)

tdf = mwf.get_codedf(mwf.counterVar)
display(tdf)

display(mwf.stimDf)

counterStatN = SimpleNamespace(minCounterVal=min(tdf.value),
                               maxCounterVal=max(tdf.value), 
                               nTotalStim=len(mwf.stimDf),
                               nUniqueStim=mwf.nstim)
ctSN = counterStatN
ctSN.counterRange = ctSN.maxCounterVal - ctSN.minCounterVal + 2 # count at both ends
ctSN.counterTickPerStim = ctSN.counterRange/ctSN.nTotalStim
display(ctSN)

0    266196329
Name: timeUs, dtype: int64

262011259

274126555

counterNPre      60
counterNPost    120
counterNStim     60
dtype: int64

,df_index,timeUs,value,timeFmStUs,timeDiffUs
0,1040,266196329,1,5914350,NaN
1,1043,266226330,2,5944351,30001.0
2,1046,266261316,3,5979337,34986.0
3,1049,266296318,4,6014339,35002.0
4,1052,266326325,5,6044346,30007.0
5,1055,266361309,6,6079330,34984.0
6,1058,266396339,7,6114360,35030.0
7,1061,266426328,8,6144349,29989.0
8,1064,266461314,9,6179335,34986.0
9,1067,266496332,10,6214353,35018.0


,stimPySelLevel,tStim1AzimuthDeg,tStim2AzimuthDeg,tStim1ElevationDeg,tStim2ElevationDeg,tStim1GratingSpeedDps,tStim2GratingSpeedDps,tStim1GratingSpatialFreqCpd,tStim2GratingSpatialFreqCpd,tStim1OriNoiseF0Cpd,...,tATrainNPulses,tBTrainNPulses,tATrainPulseLengthMs,tBTrainPulseLengthMs,tARampLengthMs,tBRampLengthMs,tARampExtraConstantLengthMs,tBRampExtraConstantLengthMs,tAStartOffsetMs,tBStartOffsetMs
0,2,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
1,5,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
2,6,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
3,7,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
4,0,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
5,4,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
6,3,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
7,1,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0


namespace(minCounterVal=1,
          maxCounterVal=1919,
          nTotalStim=8,
          nUniqueStim=8,
          counterRange=1920,
          counterTickPerStim=240.0)

In [10]:
1920/mwf.nframes_stim

8.0

In [11]:
#dir(mwf)
mwf.df.loc[0:10, :]

,tagname,timeUs,value,timeFmStUs
0,#allowAltFailover,260281979,0,0
1,#state_system_mode,260281995,0,16
2,#announceMessage,260281998,0,19
3,#stimDisplayUpdate,260282007,[],28
4,#stimDisplayCapture,260282008,,29
5,#experimentLoadProgress,260282009,1.0,30
6,#loadedExperiment,260282009,{'/Users/holdanddetect/Repositories/Experiment...,30
7,#announceSound,260282020,"{'action': 'play', 'filename': '/var/folders/8...",41
8,#announceCalibrator,260282022,{},43
9,#requestCalibrator,260282023,{},44


In [12]:
counterIx = mwf.df.tagname.isin(['counterFIO','strobedDigitalWord', 'stimPySelLevel'])
pd.set_option('display.max_rows', 50)
pd.set_option('display.min_rows', 40)


mwf.df.loc[counterIx,:]

,tagname,timeUs,value,timeFmStUs
159,stimPySelLevel,260282108,2,129
197,strobedDigitalWord,260282124,4,145
199,counterFIO,260282124,2245,145
363,stimPySelLevel,260284031,2,2052
401,counterFIO,260284135,2245,2156
402,strobedDigitalWord,260284136,4,2157
574,stimPySelLevel,262000699,2,1718720
612,counterFIO,262000750,2245,1718771
613,strobedDigitalWord,262000750,4,1718771
637,strobedDigitalWord,262009570,0,1727591


In [13]:
mwf.df.loc[(mwf.df.tagname=='stimPySelLevel'),:]

,tagname,timeUs,value,timeFmStUs
159,stimPySelLevel,260282108,2,129
363,stimPySelLevel,260284031,2,2052
574,stimPySelLevel,262000699,2,1718720
693,stimPySelLevel,262016905,2,1734926
1922,stimPySelLevel,274133434,5,13851455
2905,stimPySelLevel,282133522,6,21851543
3888,stimPySelLevel,290134206,7,29852227
4871,stimPySelLevel,298133377,0,37851398
5866,stimPySelLevel,306133609,4,45851630
6849,stimPySelLevel,314133860,3,53851881
